In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [ ]:
!pip install pytorch-lightning
!pip install pytorch-forecasting
!pip install pandas numpy matplotlib seaborn joblib tensorflow ipython pyarrow notebook
!pip install --upgrade seaborn

In [ ]:
# %% Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib 
import os
from datetime import datetime
import warnings
from IPython.display import display, Markdown

%matplotlib inline 

import torch
import lightning.pytorch as pl 
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

try:
    from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
    from pytorch_forecasting.data import GroupNormalizer, NaNLabelEncoder
    from pytorch_forecasting.metrics import MAE, SMAPE, QuantileLoss
    from pytorch_forecasting.utils import profile
    PYTORCH_FORECASTING_INSTALLED = True
    print("Successfully imported PyTorch Forecasting components.")
except ImportError as e:
    PYTORCH_FORECASTING_INSTALLED = False
    print(f"ERROR: PyTorch Forecasting Import failed: {e}. TFT functionality will be disabled.")
    print("Please install it: pip install pytorch-forecasting")

from tensorflow.keras.models import load_model 
print(f"PyTorch available: {torch.cuda.is_available()}" if PYTORCH_FORECASTING_INSTALLED and torch.cuda.is_available() else "PyTorch CUDA not available or PyTorch Forecasting not installed.")

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 7)
if PYTORCH_FORECASTING_INSTALLED: 
    pl.seed_everything(42)
else:
    import random
    random.seed(42)
    np.random.seed(42)
    if torch.cuda.is_available(): 
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)

# %% Common Configuration
EXPERIMENT_NAME_BASE = "Assembly_Anomaly_Detection_TFT"
BASE_DATA_DIR = '../data' #<-- ADJUST
NORMAL_DATA_FILE_PATH_COMMON = os.path.join(BASE_DATA_DIR, 'session_02_normal_seq125/merged_df.csv') #<-- ADJUST
CNNGRU_MODEL_PATH = '../models/best_cnnlstm_model_vel_v2.keras' #<-- ADJUST
CNN_GRU_INPUT_CONFIG = {
    'FRAME_ID_COLUMN': 'timestamp_mmwave', 'IMG_HEIGHT': 48, 'IMG_WIDTH': 48, 'NUM_CHANNELS': 2,
    'SEQUENCE_LENGTH': 5, 'X_MIN_METERS': -0.5, 'X_MAX_METERS': 0.5,
    'Y_MIN_METERS': 0.1,  'Y_MAX_METERS': 1.0, 'X_BINS': 48, 'Y_BINS': 48,
}
TFT_MAX_ENCODER_LENGTH = 24 
TFT_MAX_PREDICTION_LENGTH = 1
TFT_BATCH_SIZE = 64 
TFT_LEARNING_RATE = 1e-3
TFT_HIDDEN_SIZE = 32 
TFT_ATTENTION_HEADS = 2 
TFT_DROPOUT = 0.1
TFT_GRADIENT_CLIP_VAL = 0.1 
TFT_MAX_EPOCHS = 50 # Or fewer if it converges faster
CURRENT_EXPERIMENT_NAME = f"{EXPERIMENT_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
MAIN_OUTPUT_DIR = f"results_outputs_tft/{CURRENT_EXPERIMENT_NAME}"
PROCESSED_DATA_FOR_TFT_DIR = os.path.join(MAIN_OUTPUT_DIR, "processed_for_tft")
TFT_MODEL_CHECKPOINTS_DIR = os.path.join(MAIN_OUTPUT_DIR, "tft_checkpoints")
TFT_PLOTS_DIR = os.path.join(MAIN_OUTPUT_DIR, "plots") # For Part 2 plots
TFT_RESULTS_DIR = os.path.join(MAIN_OUTPUT_DIR, "results_data") # For Part 2 CSVs
for directory in [PROCESSED_DATA_FOR_TFT_DIR, TFT_MODEL_CHECKPOINTS_DIR, TFT_PLOTS_DIR, TFT_RESULTS_DIR]:
    os.makedirs(directory, exist_ok=True)
print(f"All experiment outputs will be saved in: {os.path.abspath(MAIN_OUTPUT_DIR)}")
PROCESSED_NORMAL_DATA_FILE = os.path.join(PROCESSED_DATA_FOR_TFT_DIR, "normal_data_for_tft.parquet")
PROCESSED_ANOMALY_FILES_PREFIX = os.path.join(PROCESSED_DATA_FOR_TFT_DIR, "anomaly_data_for_tft")
ANOMALY_DATA_FILE_PATHS = [ #<-- ADJUST THESE
    os.path.join(BASE_DATA_DIR, 'session_03_normal_seq135/merged_df.csv'),
    os.path.join(BASE_DATA_DIR, 'session_04_normal_seq145/merged_df.csv'),
    os.path.join(BASE_DATA_DIR, 'session_05_anomaly_fast/merged_df.csv'),
    os.path.join(BASE_DATA_DIR, 'session_06_anomaly_slow/merged_df.csv'),
]
ANOMALY_DATA_LABELS = ["Anomaly Seq 135", "Anomaly Seq 145", "Anomaly Fast", "Anomaly Slow"]


# %% ###############################################################################
# PART 1: DATA EXTRACTION & FEATURE ENGINEERING FOR TFT
# ###############################################################################
display(Markdown("## PART 1: Data Extraction & Feature Engineering for TFT"))

# %% Utilities (Part 1)
# ... (load_cnngru_model, create_heatmaps_from_points, etc. - same as before) ...
def load_cnngru_model(model_path):
    print(f"Loading CNN-GRU model from: {os.path.abspath(model_path)}")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"CNN-GRU model file not found: {os.path.abspath(model_path)}")
    model = load_model(model_path, compile=False) 
    print("CNN-GRU model loaded successfully.")
    return model

def create_heatmaps_from_points(df_points, config):
    all_single_frame_heatmaps = []
    unique_frame_ids_sorted = sorted(df_points[config['FRAME_ID_COLUMN']].unique())
    grouped_points = df_points.groupby(config['FRAME_ID_COLUMN'])
    for frame_id_val in unique_frame_ids_sorted:
        if frame_id_val not in grouped_points.groups:
            heatmap = np.zeros((config['Y_BINS'], config['X_BINS'], config['NUM_CHANNELS']), dtype=np.float32)
            all_single_frame_heatmaps.append(heatmap)
            continue
        points_in_frame_df = grouped_points.get_group(frame_id_val)
        current_heatmap = np.zeros((config['Y_BINS'], config['X_BINS'], config['NUM_CHANNELS']), dtype=np.float32)
        if 'x' not in points_in_frame_df.columns or 'y' not in points_in_frame_df.columns:
            all_single_frame_heatmaps.append(current_heatmap); continue
        x_coords, y_coords = points_in_frame_df['x'].to_numpy(), points_in_frame_df['y'].to_numpy()
        x_indices = np.floor((x_coords - config['X_MIN_METERS']) / (config['X_MAX_METERS'] - config['X_MIN_METERS']) * config['X_BINS']).astype(int)
        y_indices = np.floor((y_coords - config['Y_MIN_METERS']) / (config['Y_MAX_METERS'] - config['Y_MIN_METERS']) * config['Y_BINS']).astype(int)
        valid_indices = (x_indices >= 0) & (x_indices < config['X_BINS']) & (y_indices >= 0) & (y_indices < config['Y_BINS'])
        x_indices_valid, y_indices_valid = x_indices[valid_indices], y_indices[valid_indices]
        points_in_frame_valid = points_in_frame_df[valid_indices]
        if not points_in_frame_valid.empty:
            snr_values = points_in_frame_valid.get('snr', pd.Series(1.0, index=points_in_frame_valid.index)).fillna(1.0).to_numpy()
            np.maximum.at(current_heatmap[:, :, 0], (y_indices_valid, x_indices_valid), snr_values)
            if config['NUM_CHANNELS'] > 1:
                velocity_values = points_in_frame_valid.get('doppler', pd.Series(0.0, index=points_in_frame_valid.index)).fillna(0.0).to_numpy()
                temp_vel_sum = np.zeros((config['Y_BINS'], config['X_BINS']), dtype=np.float32)
                temp_vel_count = np.zeros((config['Y_BINS'], config['X_BINS']), dtype=np.int32)
                np.add.at(temp_vel_sum, (y_indices_valid, x_indices_valid), velocity_values)
                np.add.at(temp_vel_count, (y_indices_valid, x_indices_valid), 1)
                current_heatmap[:, :, 1] = np.divide(temp_vel_sum, temp_vel_count, out=np.zeros_like(temp_vel_sum), where=temp_vel_count!=0)
        all_single_frame_heatmaps.append(current_heatmap)
    if not all_single_frame_heatmaps: return np.array([]), {}
    return np.array(all_single_frame_heatmaps), {fid: i for i, fid in enumerate(unique_frame_ids_sorted)}

def create_sequences_from_heatmaps(heatmaps, sequence_length):
    if len(heatmaps) < sequence_length: return np.array([])
    return np.array([heatmaps[i : i + sequence_length] for i in range(len(heatmaps) - sequence_length + 1)])

def predict_box_interactions_cnngru(df_mmwave_points_input, cnngru_model, cnn_gru_config):
    print("Starting box prediction using CNN-GRU...")
    df_mmwave_points = df_mmwave_points_input.copy()
    if df_mmwave_points.empty: return pd.DataFrame(columns=['timestamp_numeric', 'predicted_box'])
    heatmaps_all_frames, frame_id_to_idx_map = create_heatmaps_from_points(df_mmwave_points, cnn_gru_config)
    if heatmaps_all_frames.size == 0: return pd.DataFrame(columns=['timestamp_numeric', 'predicted_box'])
    sequences = create_sequences_from_heatmaps(heatmaps_all_frames, cnn_gru_config['SEQUENCE_LENGTH'])
    if sequences.size == 0: return pd.DataFrame(columns=['timestamp_numeric', 'predicted_box'])
    sequences = sequences.astype(np.float32)
    predictions_proba = cnngru_model.predict(sequences, verbose=0)
    predicted_classes = np.argmax(predictions_proba, axis=1)
    print(f"  CNN-GRU prediction complete. Found classes: {np.unique(predicted_classes)}")
    idx_to_frame_id_map = {v: k for k, v in frame_id_to_idx_map.items()}
    used_frame_indices = sorted(idx_to_frame_id_map.keys())
    unique_frames_used_for_heatmaps = [idx_to_frame_id_map[i] for i in used_frame_indices]
    unique_frames_timestamps_df = df_mmwave_points[df_mmwave_points[cnn_gru_config['FRAME_ID_COLUMN']].isin(unique_frames_used_for_heatmaps)]
    unique_frames_timestamps_df = unique_frames_timestamps_df[[cnn_gru_config['FRAME_ID_COLUMN'], 'timestamp_numeric']].drop_duplicates(subset=[cnn_gru_config['FRAME_ID_COLUMN']]).sort_values(cnn_gru_config['FRAME_ID_COLUMN']).reset_index(drop=True)
    results_list = []
    for i in range(len(sequences)):
        last_frame_in_sequence_heatmap_idx = i + cnn_gru_config['SEQUENCE_LENGTH'] - 1
        if last_frame_in_sequence_heatmap_idx < len(unique_frames_timestamps_df):
            timestamp_val = unique_frames_timestamps_df['timestamp_numeric'].iloc[last_frame_in_sequence_heatmap_idx]
            results_list.append({'timestamp_numeric': timestamp_val, 'predicted_box': predicted_classes[i]})
    return pd.DataFrame(results_list).sort_values('timestamp_numeric').reset_index(drop=True)

def extract_state_changes_and_features(df_predicted_boxes, group_id_val="assembly_line_1"):
    if df_predicted_boxes.empty: return pd.DataFrame()
    print(f"Extracting state changes from {len(df_predicted_boxes)} records for group '{group_id_val}'...")
    changed_records = []
    last_box, last_timestamp = -999, np.nan # Use -999 as a sentinel for "no previous box"
    for _, row in df_predicted_boxes.iterrows():
        current_box, current_timestamp = row['predicted_box'], row['timestamp_numeric']
        if current_box != last_box:
            # Duration of the state that just ended
            box_duration = current_timestamp - last_timestamp if last_box != -999 and not np.isnan(last_timestamp) else 0.0 
            changed_records.append({
                'timestamp_numeric': current_timestamp, 
                'predicted_box': current_box, # This is the TARGET for TFT (the state that is starting)
                'prev_box': last_box if last_box != -999 else 0, # Previous box state, 0 if no prior
                'prev_box_duration_s': box_duration # Duration of that previous_box
            })
            last_box, last_timestamp = current_box, current_timestamp
    df_state_changes = pd.DataFrame(changed_records)
    if df_state_changes.empty: return df_state_changes
    df_state_changes['time_idx'] = range(len(df_state_changes)) # Integer sequence index for TFT
    df_state_changes['group_id'] = group_id_val 
    for col in ['predicted_box', 'prev_box']: df_state_changes[col] = df_state_changes[col].astype(int)
    df_state_changes['prev_box_duration_s'] = df_state_changes['prev_box_duration_s'].astype(float)
    print(f"  Reduced to {len(df_state_changes)} state changes for TFT input."); return df_state_changes.reset_index(drop=True)

def preprocess_raw_data(data_path, frame_id_col='timestamp_mmwave'):
    print(f"Loading raw data from: {data_path}")
    try: df = pd.read_csv(data_path)
    except FileNotFoundError: print(f"ERR: File not found: {data_path}"); return pd.DataFrame()
    original_fid_col = frame_id_col
    if frame_id_col not in df.columns: frame_id_col = 'timestamp' if 'timestamp' in df.columns else None
    if not frame_id_col: print(f"ERR: Frame ID col '{original_fid_col}' or 'timestamp' not found."); return pd.DataFrame()
    if not pd.api.types.is_datetime64_any_dtype(df[frame_id_col]): df[frame_id_col] = pd.to_datetime(df[frame_id_col], errors='coerce')
    df = df.dropna(subset=[frame_id_col])
    if df.empty: print("DataFrame empty post datetime/dropna."); return pd.DataFrame()
    df = df.sort_values(frame_id_col).reset_index(drop=True)
    df['timestamp_numeric'] = (df[frame_id_col] - df[frame_id_col].iloc[0]).dt.total_seconds()
    if original_fid_col != frame_id_col and original_fid_col not in df.columns: df[original_fid_col] = df[frame_id_col] # Ensure original FRAME_ID_COLUMN is present for heatmap creation
    print(f"  Preprocessing complete for {data_path}. Shape: {df.shape}"); return df

# %% Main Execution - Part 1
# ... (run_part1_data_extraction_for_tft function definition - same as before) ...
def run_part1_data_extraction_for_tft():
    if not PYTORCH_FORECASTING_INSTALLED: print("Skip Part 1: PyTorch Forecasting missing."); return {}
    display(Markdown("### 1.1 Loading CNN-GRU Model"))
    try: cnngru_model = load_cnngru_model(CNNGRU_MODEL_PATH)
    except Exception as e: print(f"ERR loading CNN-GRU: {e}."); return {}
    processed_paths = {"normal": None, "anomaly": [], "anomaly_labels": []}
    display(Markdown("### 1.2 Processing Normal Data"))
    df_norm_raw = preprocess_raw_data(NORMAL_DATA_FILE_PATH_COMMON, CNN_GRU_INPUT_CONFIG['FRAME_ID_COLUMN'])
    if not df_norm_raw.empty:
        df_norm_preds = predict_box_interactions_cnngru(df_norm_raw, cnngru_model, CNN_GRU_INPUT_CONFIG)
        if not df_norm_preds.empty:
            df_norm_tft = extract_state_changes_and_features(df_norm_preds, "normal_sequence")
            if not df_norm_tft.empty:
                df_norm_tft.to_parquet(PROCESSED_NORMAL_DATA_FILE); print(f"Saved normal TFT data: {PROCESSED_NORMAL_DATA_FILE}"); processed_paths["normal"] = PROCESSED_NORMAL_DATA_FILE
            else: print("No state changes from normal data.")
        else: print("CNN-GRU failed for normal data.")
    else: print("Normal data preprocessing failed.")
    display(Markdown("### 1.3 Processing Anomaly Data"))
    for i, path in enumerate(ANOMALY_DATA_FILE_PATHS):
        lbl = ANOMALY_DATA_LABELS[i]; safe_lbl = lbl.replace(' ', '_').lower(); display(Markdown(f"#### Processing: {lbl}"))
        df_anom_raw = preprocess_raw_data(path, CNN_GRU_INPUT_CONFIG['FRAME_ID_COLUMN'])
        if not df_anom_raw.empty:
            df_anom_preds = predict_box_interactions_cnngru(df_anom_raw, cnngru_model, CNN_GRU_INPUT_CONFIG)
            if not df_anom_preds.empty:
                df_anom_tft = extract_state_changes_and_features(df_anom_preds, f"anomaly_seq_{safe_lbl}") # Ensure group_id matches Part 3 expectation
                if not df_anom_tft.empty:
                    out_file = f"{PROCESSED_ANOMALY_FILES_PREFIX}_{safe_lbl}.parquet"; df_anom_tft.to_parquet(out_file); print(f"Saved anomaly TFT data ({lbl}): {out_file}"); processed_paths["anomaly"].append(out_file); processed_paths["anomaly_labels"].append(lbl)
                else: print(f"No state changes from anomaly: {lbl}")
            else: print(f"CNN-GRU failed for anomaly: {lbl}")
        else: print(f"Anomaly preprocessing failed for: {lbl}")
    display(Markdown(f"--- Part 1 Complete ---")); return processed_paths

# %% Execute Part 1
generated_tft_data_files = run_part1_data_extraction_for_tft()
print(f"\nPaths to generated Parquet files for TFT: {generated_tft_data_files}")


# %% ###############################################################################
# PART 2: TFT TRAINING & ANOMALY DETECTION
# ###############################################################################
display(Markdown("## PART 2: TFT Training & Anomaly Detection"))

# %% Utilities (Part 2 - create_tft_dataset)
# ... (create_tft_dataset definition - the one that produces >1 sample for prediction) ...
def create_tft_dataset(data: pd.DataFrame,
                       max_encoder_length_config: int, 
                       max_prediction_length_config: int,
                       existing_scalers_and_encoders=None, 
                       for_training_split: bool = False, 
                       train_cutoff_time_idx: int = None): 
    data_copy = data.copy()
    data_copy["predicted_box"] = data_copy["predicted_box"].astype(np.float32)
    if "prev_box_duration_s" in data_copy.columns: data_copy["prev_box_duration_s"] = data_copy["prev_box_duration_s"].astype(np.float32)
    if "prev_box" in data_copy.columns: data_copy["prev_box_str"] = data_copy["prev_box"].astype(str)
    else: data_copy["prev_box_str"] = "0" 
    if "group_id" not in data_copy.columns: data_copy["group_id"] = "default_group"
    target_normalizer_instance, categorical_encoders_instance = None, None
    static_categoricals_to_use, static_reals_to_use = [], [] # Initialize
    if existing_scalers_and_encoders is not None:
        target_normalizer_instance = existing_scalers_and_encoders.target_normalizer
        categorical_encoders_instance = existing_scalers_and_encoders.categorical_encoders
        static_categoricals_to_use = getattr(existing_scalers_and_encoders, 'static_categoricals', [])
        static_reals_to_use = getattr(existing_scalers_and_encoders, 'static_reals', [])
    else:
        target_normalizer_instance = GroupNormalizer(groups=["group_id"], transformation="softplus", center=False)
        categorical_encoders_instance = {"prev_box_str": NaNLabelEncoder(add_nan=True), "group_id": NaNLabelEncoder(add_nan=True)}
    data_to_process = data_copy
    if for_training_split and train_cutoff_time_idx is not None:
        data_to_process = data_copy[data_copy["time_idx"] <= train_cutoff_time_idx]
        if data_to_process.empty: raise ValueError("Training data segment empty post-cutoff.")
    min_required_length = max_encoder_length_config + max_prediction_length_config
    if len(data_to_process) < min_required_length:
        purpose = "training" if for_training_split else "prediction/validation"; print(f"Warning: Data for {purpose} (len {len(data_to_process)}) < required ({min_required_length}).")
    dataset = TimeSeriesDataSet(
        data_to_process, time_idx="time_idx", target="predicted_box", group_ids=["group_id"],
        max_encoder_length=max_encoder_length_config, min_encoder_length=max_encoder_length_config,
        max_prediction_length=max_prediction_length_config, min_prediction_length=max_prediction_length_config,
        static_categoricals=static_categoricals_to_use, static_reals=static_reals_to_use,
        time_varying_known_categoricals=[], time_varying_known_reals=[],
        time_varying_unknown_categoricals=["prev_box_str"],
        time_varying_unknown_reals=["prev_box_duration_s"] if "prev_box_duration_s" in data_copy.columns else [],
        target_normalizer=target_normalizer_instance, categorical_encoders=categorical_encoders_instance,
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
        allow_missing_timesteps=True,
    )
    return dataset

# %% Main Execution - Part 2
# ... (run_part2_tft_anomaly_detection function - same as the one that gave correct sample counts) ...
def run_part2_tft_anomaly_detection(tft_data_files_dict):
    if not PYTORCH_FORECASTING_INSTALLED: print("Skip Part 2: PyTorch Forecasting missing."); return
    all_results_data = {}; tft_model_trained = None; training_dataset_obj = None
    display(Markdown("### 2.1 Loading Processed Data for TFT"))
    normal_data_path = tft_data_files_dict.get("normal")
    if not normal_data_path or not os.path.exists(normal_data_path): print(f"ERR: Normal data file missing."); return
    try:
        df_normal_full = pd.read_parquet(normal_data_path); print(f"Loaded normal data: {normal_data_path}, shape: {df_normal_full.shape}")
        df_normal_full['predicted_box'] = df_normal_full['predicted_box'].astype(np.float32)
        if 'prev_box_duration_s' in df_normal_full.columns: df_normal_full['prev_box_duration_s'] = df_normal_full['prev_box_duration_s'].astype(np.float32)
    except Exception as e: print(f"ERR loading normal parquet: {e}"); return
    min_req_rows = TFT_MAX_ENCODER_LENGTH + TFT_MAX_PREDICTION_LENGTH
    if df_normal_full.empty or len(df_normal_full) < min_req_rows: print(f"Normal data too short ({len(df_normal_full)} vs {min_req_rows})."); return
    display(Markdown("### 2.2 Creating TimeSeriesDataSet for Training"))
    try:
        max_idx_norm = df_normal_full["time_idx"].max()
        training_dataset_obj = create_tft_dataset(df_normal_full.copy(), TFT_MAX_ENCODER_LENGTH, TFT_MAX_PREDICTION_LENGTH, for_training_split=True, train_cutoff_time_idx=max_idx_norm)
        train_dl = training_dataset_obj.to_dataloader(train=True, batch_size=TFT_BATCH_SIZE, num_workers=0, shuffle=True)
        val_dl = training_dataset_obj.to_dataloader(train=False, batch_size=TFT_BATCH_SIZE*2, num_workers=0, shuffle=False)
        print(f"Training DS samples: {len(training_dataset_obj)}, Val DS samples: {len(val_dl.dataset)}")
    except Exception as e: print(f"ERR creating training DS: {e}"); import traceback; traceback.print_exc(); return
    if len(train_dl) == 0: print("ERR: Training dataloader empty."); return
    display(Markdown("### 2.3 Defining and Training Temporal Fusion Transformer"))
    lr_logger, ckpt_cb = LearningRateMonitor(), ModelCheckpoint(dirpath=TFT_MODEL_CHECKPOINTS_DIR, filename="tft-best-{epoch}-{val_loss:.4f}", save_top_k=1, monitor="val_loss", mode="min")
    early_stop_cb = EarlyStopping(monitor="val_loss", min_delta=1e-5, patience=10, verbose=True, mode="min")
    trainer = pl.Trainer(max_epochs=TFT_MAX_EPOCHS, accelerator="auto", enable_model_summary=True, gradient_clip_val=TFT_GRADIENT_CLIP_VAL, callbacks=[lr_logger, ckpt_cb, early_stop_cb], logger=TensorBoardLogger(save_dir=os.path.join(MAIN_OUTPUT_DIR, "tb_logs"), name="tft_model"))
    tft_model_instance = TemporalFusionTransformer.from_dataset(training_dataset_obj, learning_rate=TFT_LEARNING_RATE, hidden_size=TFT_HIDDEN_SIZE, attention_head_size=TFT_ATTENTION_HEADS, dropout=TFT_DROPOUT, hidden_continuous_size=TFT_HIDDEN_SIZE//2, output_size=1, loss=QuantileLoss(quantiles=[0.5]), log_interval=10, reduce_on_plateau_patience=4)
    print(f"Training TFT model for {TFT_MAX_EPOCHS} epochs...")
    try:
        trainer.fit(tft_model_instance, train_dl, val_dl); print("TFT Training complete."); best_path = ckpt_cb.best_model_path
        if best_path and os.path.exists(best_path): tft_model_trained = TemporalFusionTransformer.load_from_checkpoint(best_path); print(f"Loaded best model: {best_path}")
        else: print("No best model. Using last trained model."); tft_model_trained = tft_model_instance
    except Exception as e: print(f"ERR during training: {e}"); import traceback; traceback.print_exc(); return
    if tft_model_trained is None: print("TFT model not trained."); return
    display(Markdown("### 2.4 Anomaly Detection: Predicting and Calculating Errors"))
    print("\nPredicting on Normal data..."); df_norm_eval = df_normal_full.copy()
    try:
        norm_pred_ds = create_tft_dataset(df_norm_eval, TFT_MAX_ENCODER_LENGTH, TFT_MAX_PREDICTION_LENGTH, training_dataset_obj, False)
        norm_pred_dl = norm_pred_ds.to_dataloader(train=False, batch_size=TFT_BATCH_SIZE*2, num_workers=0, shuffle=False)
        print(f"--- DEBUG: Normal - Samples: {len(norm_pred_ds)}, Batches: {len(norm_pred_dl)}")
        if len(norm_pred_dl) == 0: raise ValueError("Normal pred dataloader empty")
        preds_norm_tpl = tft_model_trained.predict(norm_pred_dl, mode="prediction", return_x=True)
        preds_norm_tensor, x_ret_norm = preds_norm_tpl[0], preds_norm_tpl[1]; preds_norm_pts = preds_norm_tensor.cpu().numpy().flatten()
        print(f"--- DEBUG: Normal - Num preds: {len(preds_norm_pts)}, x_ret_norm decoder_time_idx shape: {x_ret_norm['decoder_time_idx'].shape if x_ret_norm and 'decoder_time_idx' in x_ret_norm else 'None'}")
        time_idx_norm_df = x_ret_norm['decoder_time_idx'][:, 0].cpu().tolist() if x_ret_norm and 'decoder_time_idx' in x_ret_norm and x_ret_norm['decoder_time_idx'].shape[0] == len(preds_norm_pts) else [idx for xb, _ in norm_pred_dl for idx in xb['decoder_time_idx'][:,0].cpu().tolist()][:len(preds_norm_pts)]
        df_preds_out_norm = pd.DataFrame({'time_idx': time_idx_norm_df, 'predicted_box_tft': preds_norm_pts})
        df_preds_out_norm = pd.merge(df_preds_out_norm, df_norm_eval[['time_idx', 'group_id', 'predicted_box']], on='time_idx', how='left').rename(columns={'predicted_box':'actual_box_for_pred'}) # Added group_id
        df_preds_out_norm.dropna(subset=['actual_box_for_pred'], inplace=True)
        df_norm_merged = pd.merge(df_norm_eval, df_preds_out_norm, on=["time_idx", "group_id"], how="inner", suffixes=('', '_todrop')).filter(regex='^(?!.*_todrop$)') # Added group_id
        print(f"--- DEBUG: Normal - Merged shape: {df_norm_merged.shape}")
        if not df_norm_merged.empty and 'actual_box_for_pred' in df_norm_merged.columns:
            df_norm_merged['anomaly_score'] = np.abs(df_norm_merged['actual_box_for_pred'] - df_norm_merged['predicted_box_tft'])
            all_results_data["Normal_Data_Eval"] = df_norm_merged; print(f"Normal Data - MAE: {df_norm_merged['anomaly_score'].mean():.4f}")
            df_norm_merged.to_csv(os.path.join(TFT_RESULTS_DIR, "normal_data_predictions.csv"), index=False)
        else: print("Warn: Normal results empty/missing cols."); all_results_data["Normal_Data_Eval"] = pd.DataFrame()
    except Exception as e: print(f"ERR pred on normal data: {e}"); import traceback; traceback.print_exc()
    for i, anom_path in enumerate(tft_data_files_dict.get("anomaly", [])):
        label = tft_data_files_dict.get("anomaly_labels", [])[i]; print(f"\nPredicting on: {label}...")
        try:
            df_anom_full = pd.read_parquet(anom_path)
            df_anom_full['predicted_box'] = df_anom_full['predicted_box'].astype(np.float32)
            if 'prev_box_duration_s' in df_anom_full.columns: df_anom_full['prev_box_duration_s'] = df_anom_full['prev_box_duration_s'].astype(np.float32)
            if df_anom_full.empty or len(df_anom_full) < min_req_rows: print(f"{label} too short. Skip."); continue
            anom_pred_ds = create_tft_dataset(df_anom_full.copy(), TFT_MAX_ENCODER_LENGTH, TFT_MAX_PREDICTION_LENGTH, training_dataset_obj, False)
            anom_pred_dl = anom_pred_ds.to_dataloader(train=False, batch_size=TFT_BATCH_SIZE*2, num_workers=0, shuffle=False)
            print(f"--- DEBUG: Anomaly ({label}) - Samples: {len(anom_pred_ds)}, Batches: {len(anom_pred_dl)}")
            if len(anom_pred_dl) == 0: print(f"Warn: Dataloader {label} empty. Skip."); continue
            preds_anom_tpl = tft_model_trained.predict(anom_pred_dl, mode="prediction", return_x=True)
            preds_anom_tensor, x_ret_anom = preds_anom_tpl[0], preds_anom_tpl[1]; preds_anom_pts = preds_anom_tensor.cpu().numpy().flatten(); print(f"--- DEBUG: Anomaly ({label}) - Num preds: {len(preds_anom_pts)}")
            time_idx_anom_df = x_ret_anom['decoder_time_idx'][:, 0].cpu().tolist() if x_ret_anom and 'decoder_time_idx' in x_ret_anom and x_ret_anom['decoder_time_idx'].shape[0] == len(preds_anom_pts) else [idx for xb, _ in anom_pred_dl for idx in xb['decoder_time_idx'][:,0].cpu().tolist()][:len(preds_anom_pts)]
            df_preds_out_anom = pd.DataFrame({'time_idx': time_idx_anom_df, 'predicted_box_tft': preds_anom_pts})
            df_preds_out_anom = pd.merge(df_preds_out_anom, df_anom_full[['time_idx', 'group_id', 'predicted_box']], on='time_idx', how='left').rename(columns={'predicted_box':'actual_box_for_pred'}) # Added group_id
            df_preds_out_anom.dropna(subset=['actual_box_for_pred'], inplace=True)
            df_anom_merged = pd.merge(df_anom_full, df_preds_out_anom, on=["time_idx", "group_id"], how="inner", suffixes=('', '_todrop')).filter(regex='^(?!.*_todrop$)') # Added group_id
            print(f"--- DEBUG: Anomaly ({label}) - Merged shape: {df_anom_merged.shape}")
            if not df_anom_merged.empty and 'actual_box_for_pred' in df_anom_merged.columns:
                df_anom_merged['anomaly_score'] = np.abs(df_anom_merged['actual_box_for_pred'] - df_anom_merged['predicted_box_tft'])
                all_results_data[label] = df_anom_merged; print(f"Anomaly Data ({label}) - MAE: {df_anom_merged['anomaly_score'].mean():.4f}")
                df_anom_merged.to_csv(os.path.join(TFT_RESULTS_DIR, f"anomaly_data_{label.replace(' ', '_').lower()}_predictions.csv"), index=False)
            else: print(f"Warn: Anomaly results {label} empty/missing cols."); all_results_data[label] = pd.DataFrame()
        except Exception as e: print(f"ERR pred on anomaly {label}: {e}"); import traceback; traceback.print_exc()
    # Plotting from Part 2 (should work now)
    print("\n--- DEBUGGING all_results_data (just before plotting Part 2 style) ---")
    for key, df_result in all_results_data.items():
        if df_result is not None: print(f"Key: {key}, Shape: {df_result.shape}, Columns: {df_result.columns.tolist() if not df_result.empty else 'Empty DF'}")
        else: print(f"Key: {key}, DataFrame is None.")
    print("--- END DEBUGGING ---\n")
    display(Markdown("### 2.5 Visualizing Anomaly Scores (from Part 2 Logic)"))
    if all_results_data:
        plt.figure(figsize=(15, 7)); any_data_plotted_dist = False
        for lbl_dist, df_res_dist in all_results_data.items():
            if df_res_dist is not None and not df_res_dist.empty and 'anomaly_score' in df_res_dist.columns and len(df_res_dist['anomaly_score'].dropna()) > 1:
                sns.kdeplot(df_res_dist['anomaly_score'], label=f"{lbl_dist} (MAE)", fill=True, alpha=0.5); any_data_plotted_dist = True
            elif df_res_dist is not None and not df_res_dist.empty and 'anomaly_score' in df_res_dist.columns: print(f"Skipping KDE for {lbl_dist}, only {len(df_res_dist['anomaly_score'].dropna())} valid points.")
        if any_data_plotted_dist:
            plt.title("Distribution of State Anomaly Scores (TFT)"); plt.xlabel("State Anomaly Score (MAE)"); plt.ylabel("Density"); plt.legend(); plt.grid(True)
            plot_path_dist = os.path.join(TFT_PLOTS_DIR, "state_anomaly_score_distributions.eps"); plt.savefig(plot_path_dist, format='eps'); print(f"Saved State Anomaly KDE plot: {plot_path_dist}"); plt.show()
        else: print("No data for State Anomaly KDE distributions.")
        for lbl_viz, df_res_viz in all_results_data.items():
            if df_res_viz is not None and not df_res_viz.empty and all(c in df_res_viz.columns for c in ['anomaly_score','timestamp_numeric','actual_box_for_pred','predicted_box_tft']):
                if len(df_res_viz) < 2: print(f"Skipping State TS for {lbl_viz}, <2 data points."); continue
                fig_ts, ax1_ts = plt.subplots(figsize=(18, 6))
                ax1_ts.plot(df_res_viz['timestamp_numeric'], df_res_viz['actual_box_for_pred'], label='Actual Box', color='tab:blue', ls='--', marker='o', ms=3, alpha=0.7)
                ax1_ts.plot(df_res_viz['timestamp_numeric'], df_res_viz['predicted_box_tft'], label='TFT Predicted Box', color='tab:green', ls=':', marker='x', ms=3, alpha=0.7)
                ax1_ts.set_xlabel("Timestamp"); ax1_ts.set_ylabel("Box State", color='tab:blue'); ax1_ts.tick_params(axis='y', labelcolor='tab:blue'); ax1_ts.legend(loc='upper left')
                ax2_ts = ax1_ts.twinx()
                ax2_ts.plot(df_res_viz['timestamp_numeric'], df_res_viz['anomaly_score'], label='State Anomaly Score', color='tab:red', alpha=0.6, lw=1.5)
                ax2_ts.set_ylabel('State Anomaly Score (MAE)', color='tab:red'); ax2_ts.tick_params(axis='y', labelcolor='tab:red'); ax2_ts.legend(loc='upper right')
                plt.title(f"State Predictions & Anomaly Scores: {lbl_viz}"); plt.grid(True, ls=':', alpha=0.5); fig_ts.tight_layout()
                plot_path_ts = os.path.join(TFT_PLOTS_DIR, f"state_ts_scores_{lbl_viz.replace(' ', '_').lower()}.eps"); plt.savefig(plot_path_ts, format='eps'); print(f"Saved State TS plot for {lbl_viz}: {plot_path_ts}"); plt.show()
            else: print(f"Skipping State TS for {lbl_viz}, missing cols/empty data.")
    else: print("No results from Part 2 to plot.")
    display(Markdown(f"--- Part 2 Complete ---"))

# %% Execute Part 2
# This cell should be run after Part 1 has generated `generated_tft_data_files`
if 'generated_tft_data_files' in locals() and \
   generated_tft_data_files and \
   generated_tft_data_files.get("normal"):
    run_part2_tft_anomaly_detection(generated_tft_data_files)
else:
    display(Markdown("## Skipping Part 2: TFT Anomaly Detection"))
    print("Reason: `generated_tft_data_files` from Part 1 is missing or lacks normal data path.")



In [ ]:
# %% ###############################################################################
# PART 3: ANOMALY DETECTION IN STATE CHANGE DURATIONS
# This cell assumes Part 1 and Part 2 have run and `generated_tft_data_files`,
# `MAIN_OUTPUT_DIR`, `TFT_RESULTS_DIR`, and relevant global configs are defined.
# ###############################################################################
display(Markdown("## PART 3: Anomaly Detection in State Change Durations"))

# %% Utilities (Part 3)
def calculate_normal_duration_stats(df_normal_tft_data: pd.DataFrame, 
                                    min_samples_for_stats: int = 3, 
                                    default_std_epsilon: float = 1e-6):
    """
    Calculates mean and standard deviation of 'prev_box_duration_s' 
    for each 'prev_box' from the normal data.
    Excludes the very first state change of each group if its duration is 0.
    """
    if df_normal_tft_data.empty or not all(col in df_normal_tft_data.columns for col in ['prev_box', 'prev_box_duration_s', 'time_idx', 'group_id']):
        print("Warning: Normal TFT data for duration stats is empty or missing required columns ('prev_box', 'prev_box_duration_s', 'time_idx', 'group_id').")
        return pd.DataFrame(columns=['prev_box', 'mean_duration', 'std_duration', 'count'])

    # Exclude the first record of each group if its duration is 0 (as it's not a 'previous state' duration)
    # This assumes time_idx is 0 for the first record within each group from Part 1.
    # We only want to calculate stats on actual, measured durations.
    df_for_stats = df_normal_tft_data[~((df_normal_tft_data['time_idx'] == 0) & (df_normal_tft_data['prev_box_duration_s'] == 0.0))].copy()
    
    if df_for_stats.empty:
        print("Warning: No valid records remaining for duration stats after filtering initial state records (time_idx=0, duration=0).")
        return pd.DataFrame(columns=['prev_box', 'mean_duration', 'std_duration', 'count'])
    
    duration_stats = df_for_stats.groupby('prev_box')['prev_box_duration_s'].agg(
        mean_duration='mean', std_duration='std', count='count'
    ).reset_index()

    duration_stats['std_duration'] = np.maximum(duration_stats['std_duration'].fillna(default_std_epsilon), default_std_epsilon)
    reliable_stats = duration_stats[duration_stats['count'] >= min_samples_for_stats].copy()
    
    print(f"Calculated reliable duration statistics for {len(reliable_stats)} 'prev_box' types (min_samples_for_stats={min_samples_for_stats}).")
    if len(reliable_stats) < duration_stats['prev_box'].nunique():
        excluded_count = duration_stats['prev_box'].nunique() - len(reliable_stats)
        print(f"  Excluded {excluded_count} 'prev_box' types due to having fewer than {min_samples_for_stats} occurrences in normal data (after filtering initial states).")
    return reliable_stats

def score_duration_anomalies(df_tft_data: pd.DataFrame, 
                             normal_duration_stats: pd.DataFrame,
                             default_std_epsilon: float = 1e-6):
    """
    Scores duration anomalies based on normal stats using Z-score.
    Adds 'duration_z_score', 'mean_duration', 'std_duration', 'count' (from stats) to the DataFrame.
    """
    output_cols = ['duration_z_score', 'mean_duration', 'std_duration', 'count']
    if df_tft_data.empty:
        return df_tft_data.assign(**{col: np.nan for col in output_cols})
    if normal_duration_stats.empty:
        print("Warning: Normal duration stats are empty. Cannot score duration anomalies effectively.")
        return df_tft_data.assign(**{col: np.nan for col in output_cols})

    original_index = df_tft_data.index
    df_scored = pd.merge(df_tft_data.reset_index(drop=True), normal_duration_stats, on='prev_box', how='left')
    safe_std_duration = np.maximum(df_scored['std_duration'].fillna(default_std_epsilon), default_std_epsilon)
    df_scored['duration_z_score'] = (df_scored['prev_box_duration_s'] - df_scored['mean_duration']) / safe_std_duration
    df_scored.index = original_index
    return df_scored

# %% Main Execution - Part 3 (with Feedback Generation, Merge Debugging, and Focused Reporting)
def run_part3_duration_anomaly_detection(processed_tft_files_dict: dict, 
                                         part2_results_dir: str,
                                         part3_results_dir: str,
                                         part3_plots_dir: str,
                                         state_anomaly_threshold: float = 0.5, 
                                         duration_z_threshold: float = 1.0):
    if not PYTORCH_FORECASTING_INSTALLED: print("Skip Part 3: PyTorch Forecasting missing."); return # Should be caught earlier
    os.makedirs(part3_results_dir, exist_ok=True); os.makedirs(part3_plots_dir, exist_ok=True)
    feedback_log = []
    display(Markdown("### 3.1 Loading Normal Data and Calculating Duration Statistics"))
    normal_data_tft_path = processed_tft_files_dict.get("normal")
    if not normal_data_tft_path or not os.path.exists(normal_data_tft_path): feedback_log.append(f"ERROR: Normal TFT data file (from Part 1) not found at '{normal_data_tft_path}'. Part 3 cannot proceed."); print("\n".join(feedback_log)); return
    try:
        df_normal_tft_full = pd.read_parquet(normal_data_tft_path)
        feedback_log.append(f"NORMAL_DATA_PART1_INFO:\n  Input_Parquet_Path: {normal_data_tft_path}\n  Num_State_Changes: {len(df_normal_tft_full)}")
    except Exception as e: feedback_log.append(f"ERROR: Could not load normal Parquet file '{normal_data_tft_path}': {e}"); print("\n".join(feedback_log)); return
    if df_normal_tft_full.empty or 'prev_box_duration_s' not in df_normal_tft_full.columns: feedback_log.append("ERROR: Normal TFT data is empty or missing 'prev_box_duration_s'. Cannot calculate duration stats."); print("\n".join(feedback_log)); return
    
    normal_duration_stats = calculate_normal_duration_stats(df_normal_tft_full, min_samples_for_stats=3)
    feedback_log.append(f"DURATION_STATS_CALCULATION:\n  Status: {'Success' if not normal_duration_stats.empty else 'Failed or no reliable stats found'}.")
    if not normal_duration_stats.empty: feedback_log.extend([f"  Reliable_Stats_For_Prev_Boxes_Count: {len(normal_duration_stats)}", f"  Prev_Boxes_With_Stats: {normal_duration_stats['prev_box'].tolist()}"])
    
    all_combined_results_for_plotting = {} 
    
    datasets_to_process = [("Normal_Data_Eval", normal_data_tft_path, "normal_data_predictions.csv", "normal_sequence")]
    # Add anomaly files from the global configuration
    global ANOMALY_DATA_FILE_PATHS, ANOMALY_DATA_LABELS # Make sure these are accessible
    for i, anom_label in enumerate(ANOMALY_DATA_LABELS): # Use ANOMALY_DATA_LABELS to iterate
        safe_label = anom_label.replace(' ', '_').lower()
        # Construct path to Part 1 output for this anomaly
        # This assumes PROCESSED_ANOMALY_FILES_PREFIX is the base path and then _{safe_label}.parquet is appended
        part1_anom_path = f"{PROCESSED_ANOMALY_FILES_PREFIX}_{safe_label}.parquet" # Path from Part 1 output
        part2_anom_filename = f"anomaly_data_{safe_label}_predictions.csv" # Filename from Part 2 output
        # The group_id must match what was assigned in Part 1's extract_state_changes_and_features
        expected_anom_group_id = f"anomaly_seq_{safe_label}" 
        datasets_to_process.append((f"Anomaly_{safe_label}", part1_anom_path, part2_anom_filename, expected_anom_group_id))

    for data_label, part1_path, part2_filename, expected_group_id in datasets_to_process:
        current_feedback = [f"\n--- FEEDBACK FOR {data_label} ---"]; current_feedback.append("PART_1_DATA_INFO:")
        if not os.path.exists(part1_path): current_feedback.append(f"  ERROR: Part 1 Parquet file not found: {part1_path}"); feedback_log.extend(current_feedback); print(f"Skipping {data_label} due to missing Part 1 file."); continue
        try:
            df_current_tft_part1 = pd.read_parquet(part1_path); current_feedback.extend([f"  Input_Parquet_Path: {part1_path}", f"  Num_State_Changes_In_File: {len(df_current_tft_part1)}"])
        except Exception as e: current_feedback.append(f"  ERROR: Could not load Part 1 Parquet {part1_path}: {e}"); feedback_log.extend(current_feedback); print(f"Skipping {data_label} due to load error."); continue
        if df_current_tft_part1.empty: current_feedback.append("  Status: Empty DataFrame from Part 1. Skipping."); feedback_log.extend(current_feedback); print(f"Skipping {data_label}, empty Part 1 DF."); continue

        df_with_duration_score = score_duration_anomalies(df_current_tft_part1.copy(), normal_duration_stats)
        current_feedback.append("PART_3_DURATION_ANOMALY_INFO:")
        if 'duration_z_score' in df_with_duration_score.columns:
            valid_z = df_with_duration_score['duration_z_score'].dropna(); current_feedback.append(f"  Num_Rows_With_Duration_Z_Score: {len(valid_z)}")
            if not valid_z.empty: abs_z = valid_z.abs(); current_feedback.extend([f"  Mean_Abs_Duration_Z_Score: {abs_z.mean():.4f}", f"  Std_Dev_Abs_Duration_Z_Score: {abs_z.std():.4f}", f"  Max_Abs_Duration_Z_Score: {abs_z.max():.4f}", f"  Percent_Durations_Above_Z={duration_z_threshold}: {(abs_z > duration_z_threshold).mean()*100:.2f}%"])
            else: current_feedback.append("  No valid duration Z-scores to summarize.")
        else: current_feedback.append("  'duration_z_score' column not generated.")
        
        df_final_combined_for_dataset = df_with_duration_score.copy()
        df_final_combined_for_dataset['state_anomaly_score'] = np.nan 

        current_feedback.append("PART_2_TFT_STATE_ANOMALY_INFO:")
        part2_results_path = os.path.join(part2_results_dir, part2_filename)
        if os.path.exists(part2_results_path):
            try:
                df_part2_results_loaded = pd.read_csv(part2_results_path)
                current_feedback.extend([f"  Part2_Results_Path: {part2_results_path}", f"  Num_Rows_In_Part2_File: {len(df_part2_results_loaded)}"])
                print(f"    DEBUG: Columns in loaded Part 2 CSV ({part2_filename}): {df_part2_results_loaded.columns.tolist()}")
                cols_to_select_from_part2 = ['time_idx']; merge_on_keys = ['time_idx']
                if 'group_id' in df_part2_results_loaded.columns: cols_to_select_from_part2.append('group_id')
                source_anomaly_col_name, target_anomaly_col_name = 'anomaly_score', 'state_anomaly_score'
                if source_anomaly_col_name in df_part2_results_loaded.columns:
                    cols_to_select_from_part2.append(source_anomaly_col_name)
                    df_part2_prepared_for_merge = df_part2_results_loaded[cols_to_select_from_part2].copy()
                    df_part2_prepared_for_merge.rename(columns={source_anomaly_col_name: target_anomaly_col_name}, inplace=True)
                else:
                    current_feedback.append(f"  WARNING: Column '{source_anomaly_col_name}' (TFT state error) not found in {part2_filename}.")
                    df_part2_prepared_for_merge = df_part2_results_loaded[cols_to_select_from_part2].copy() if cols_to_select_from_part2 else pd.DataFrame(columns=cols_to_select_from_part2)
                
                print(f"\n--- DETAILED MERGE DEBUG FOR {data_label} ---")
                print(f"  df_left (df_final_combined_for_dataset before Part2 merge) shape: {df_final_combined_for_dataset.shape}")
                if not df_final_combined_for_dataset.empty: print(f"    df_left head (time_idx, group_id):\n{df_final_combined_for_dataset[['time_idx', 'group_id']].head()}")
                print(f"  df_right (df_part2_prepared_for_merge pre-group-filter) shape: {df_part2_prepared_for_merge.shape}")
                if not df_part2_prepared_for_merge.empty and target_anomaly_col_name in df_part2_prepared_for_merge.columns: print(f"    df_right head (time_idx, group_id, {target_anomaly_col_name}):\n{df_part2_prepared_for_merge[['time_idx', 'group_id', target_anomaly_col_name] if 'group_id' in df_part2_prepared_for_merge.columns else ['time_idx', target_anomaly_col_name]].head()}")
                
                df_right_actual_merge_candidate = df_part2_prepared_for_merge.copy()
                if 'group_id' in df_final_combined_for_dataset.columns and 'group_id' in df_right_actual_merge_candidate.columns:
                    df_right_actual_merge_candidate = df_right_actual_merge_candidate[df_right_actual_merge_candidate['group_id'] == expected_group_id].copy()
                    print(f"  df_right_actual_merge_candidate shape (post-group-filter for '{expected_group_id}'): {df_right_actual_merge_candidate.shape}")
                    if not df_right_actual_merge_candidate.empty: merge_on_keys.append('group_id')
                    else: print(f"    Warning: df_right became empty after group_id filter for '{expected_group_id}'.")
                print(f"  Attempting merge on keys: {merge_on_keys}")
                can_merge = True
                if not df_right_actual_merge_candidate.empty:
                    for key_col in merge_on_keys:
                        if key_col not in df_final_combined_for_dataset.columns or key_col not in df_right_actual_merge_candidate.columns: print(f"    ERROR: Merge key '{key_col}' missing."); can_merge = False; break
                        if df_final_combined_for_dataset[key_col].dtype != df_right_actual_merge_candidate[key_col].dtype:
                            print(f"    INFO: Dtype mismatch for '{key_col}'. Left: {df_final_combined_for_dataset[key_col].dtype}, Right: {df_right_actual_merge_candidate[key_col].dtype}. Casting right.")
                            try: df_right_actual_merge_candidate[key_col] = df_right_actual_merge_candidate[key_col].astype(df_final_combined_for_dataset[key_col].dtype)
                            except Exception as cast_err: print(f"    ERROR: Casting '{key_col}' failed: {cast_err}"); can_merge = False; break
                        common_vals = pd.Series(list(set(df_final_combined_for_dataset[key_col].unique()) & set(df_right_actual_merge_candidate[key_col].unique())))
                        print(f"    Key '{key_col}': Common values count = {len(common_vals)}. Left unique: {df_final_combined_for_dataset[key_col].nunique()}, Right unique: {df_right_actual_merge_candidate[key_col].nunique() if not df_right_actual_merge_candidate.empty else 0 }") # Added check for empty
                        if len(common_vals) == 0 and key_col == 'time_idx': print(f"    !!!! CRITICAL: No common 'time_idx' values for {data_label} !!!!"); can_merge = False; break
                else: can_merge = False
                
                if can_merge and not df_right_actual_merge_candidate.empty and target_anomaly_col_name in df_right_actual_merge_candidate.columns:
                    df_final_combined_for_dataset = pd.merge(df_final_combined_for_dataset.drop(columns=[target_anomaly_col_name], errors='ignore'), df_right_actual_merge_candidate, on=merge_on_keys, how='left')
                    merged_score_count = df_final_combined_for_dataset[target_anomaly_col_name].notna().sum()
                    print(f"  Shape after merge: {df_final_combined_for_dataset.shape}. Non-NaN state_anomaly_score rows: {merged_score_count}")
                elif df_right_actual_merge_candidate.empty: current_feedback.append(f"  Warning: Part 2 data for group '{expected_group_id}' empty post-filter or no common keys for merge.")
                else: current_feedback.append(f"  Warning: '{target_anomaly_col_name}' not in Part 2 data to merge after processing for {data_label}.")
                
                valid_state_scores = df_final_combined_for_dataset['state_anomaly_score'].dropna()
                current_feedback.append(f"  Num_Rows_With_State_Anomaly_Score_Post_Merge: {len(valid_state_scores)}")
                if not valid_state_scores.empty: current_feedback.extend([f"  Mean_State_Anomaly_Score: {valid_state_scores.mean():.4f}", f"  Std_Dev_State_Anomaly_Score: {valid_state_scores.std():.4f}", f"  Percent_States_Above_Threshold_{state_anomaly_threshold}: {(valid_state_scores > state_anomaly_threshold).mean()*100:.2f}%"])
                else: current_feedback.append("  No valid state anomaly scores post-merge to summarize.")
            except KeyError as ke: current_feedback.append(f"  KEY_ERROR during Part 2 merge for {data_label}: {ke}."); import traceback; traceback.print_exc() 
            except Exception as e: current_feedback.append(f"  UNEXPECTED_ERROR loading/merging Part 2 for {data_label}: {e}."); import traceback; traceback.print_exc()
        else: current_feedback.append(f"  Part2_Results_Path_Not_Found: {part2_results_path}.")
        
        current_feedback.append("COMBINED_ANALYSIS:")
        df_both_scores = df_final_combined_for_dataset.dropna(subset=['state_anomaly_score', 'duration_z_score'])
        current_feedback.append(f"  Num_Rows_With_Both_Scores_Valid: {len(df_both_scores)}")
        if len(df_both_scores) > 1:
            correlation = df_both_scores['state_anomaly_score'].corr(df_both_scores['duration_z_score'].abs()); current_feedback.append(f"  Correlation_State_vs_Abs_Duration_Z_Score: {correlation:.4f}")
            anom_by_state = df_both_scores['state_anomaly_score'] > state_anomaly_threshold; anom_by_duration = df_both_scores['duration_z_score'].abs() > duration_z_threshold
            num_anom_both = (anom_by_state & anom_by_duration).sum(); current_feedback.append(f"  Num_Anomalous_By_State_AND_Duration (State>{state_anomaly_threshold}, |Z|>{duration_z_threshold}): {num_anom_both} ({ (num_anom_both/len(df_both_scores)*100 if len(df_both_scores)>0 else 0):.2f}%)")
            # Add State OR Duration, State Only, Dur Only
            stats_perc_anom_state_or_dur = (anom_by_state | anom_by_duration).mean() * 100 if len(df_both_scores) > 0 else 0
            stats_perc_anom_state_only = (anom_by_state & ~anom_by_duration).mean() * 100 if len(df_both_scores) > 0 else 0
            stats_perc_anom_dur_only = (~anom_by_state & anom_by_duration).mean() * 100 if len(df_both_scores) > 0 else 0
            current_feedback.append(f"  Percent_Anomalous_By_State_OR_Duration: {stats_perc_anom_state_or_dur:.2f}%")
            current_feedback.append(f"  Percent_Anomalous_By_State_Only: {stats_perc_anom_state_only:.2f}%")
            current_feedback.append(f"  Percent_Anomalous_By_Duration_Only: {stats_perc_anom_dur_only:.2f}%")
        else: current_feedback.append("  Not enough data for combined correlation/AND/OR analysis.")
            
        safe_data_label_for_file = data_label.replace(' ', '_').lower()
        df_final_combined_for_dataset.to_csv(os.path.join(part3_results_dir, f"{safe_data_label_for_file}_combined_scores.csv"), index=False)
        all_combined_results_for_plotting[data_label] = df_final_combined_for_dataset; feedback_log.extend(current_feedback)
    
    print("\n\n" + "="*80 + "\nFINAL FEEDBACK LOG:\n" + "="*80 + "\n" + "\n".join(feedback_log)) 

    # --- Visualization for Part 3 (with selective plotting) ---
    display(Markdown("### 3.4 Visualizing Combined Anomaly Scores"))
    if all_combined_results_for_plotting:
        
        # --- KDE Plot for State Anomaly Scores (Focused) ---
        datasets_for_state_kde = ["Normal_Data_Eval", "Anomaly_anomaly_seq_135", "Anomaly_anomaly_seq_145"] # As per your focus
        plt.figure(figsize=(15, 7))
        any_state_kde_plotted = False
        for key, df_res in all_combined_results_for_plotting.items():
            if key in datasets_for_state_kde and df_res is not None and not df_res.empty and \
               'state_anomaly_score' in df_res.columns and df_res['state_anomaly_score'].dropna().nunique() > 1:
                sns.kdeplot(df_res['state_anomaly_score'].dropna(), label=f"{key.replace('_', ' ')} (State MAE)", fill=True, alpha=0.5) # Cleaner labels
                any_state_kde_plotted = True
        if any_state_kde_plotted:
            plt.title("Distribution of State Anomaly Scores (TFT - Sequence Focus)")
            plt.xlabel("State Anomaly Score (MAE)"); plt.ylabel("Density"); plt.legend(); plt.grid(True)
            plot_path_state_kde = os.path.join(part3_plots_dir, "state_anomaly_focused_kde.eps")
            plt.savefig(plot_path_state_kde, format='eps'); print(f"Saved focused State Anomaly KDE plot to {plot_path_state_kde}")
            plt.show()
        else:
            print("No data for focused State Anomaly KDE.")

        # --- KDE plot for Duration Z-Scores (Focused) ---
        datasets_for_duration_kde = ["Normal_Data_Eval", "Anomaly_anomaly_fast", "Anomaly_anomaly_slow"] # As per your focus
        plt.figure(figsize=(15, 7))
        any_duration_kde_plotted = False
        for key, df_res in all_combined_results_for_plotting.items():
            if key in datasets_for_duration_kde and df_res is not None and not df_res.empty and \
               'duration_z_score' in df_res.columns and df_res['duration_z_score'].dropna().abs().nunique() > 1 :
                sns.kdeplot(df_res['duration_z_score'].dropna().abs(), label=f"{key.replace('_', ' ')} (Abs Z)", fill=True, alpha=0.5, cut=0) # Cleaner labels
                any_duration_kde_plotted = True
        if any_duration_kde_plotted:
            plt.title("Distribution of Absolute Duration Z-Scores (Timing Focus)")
            plt.xlabel("Absolute Z-Score of Previous State Duration"); plt.ylabel("Density"); plt.legend(); plt.grid(True)
            plot_path_dur_kde = os.path.join(part3_plots_dir, "duration_zscore_focused_kde.eps")
            plt.savefig(plot_path_dur_kde, format='eps'); print(f"Saved focused duration Z-score distribution plot to {plot_path_dur_kde}")
            plt.show()
        else:
            print("No sufficient data to plot focused KDE for duration Z-scores.")

        # --- Individual Combined Time Series Plots for EACH Dataset ---
        # This loop generates one figure per dataset in all_combined_results_for_plotting
        figure_references_ts = {} # To store paths for paper reference

        for key, df_res_viz in all_combined_results_for_plotting.items():
            if df_res_viz is not None and not df_res_viz.empty and \
               all(c in df_res_viz.columns for c in ['timestamp_numeric', 'predicted_box', 'state_anomaly_score', 'duration_z_score']) and \
               len(df_res_viz) >= 2:

                fig, ax1 = plt.subplots(figsize=(18, 7)) # Slightly adjusted height for 3 y-axes
                
                # Plot actual box states
                color_box = 'tab:blue'
                ax1.set_xlabel("Timestamp (Numeric)")
                ax1.set_ylabel("Actual Box State (CNN-GRU)", color=color_box)
                ax1.plot(df_res_viz['timestamp_numeric'], df_res_viz['predicted_box'], label='Actual Box State', color=color_box, linestyle='-', marker='o', markersize=3, alpha=0.7)
                ax1.tick_params(axis='y', labelcolor=color_box)
                ax1.legend(loc='upper left')
                # Add horizontal lines for thresholds on the score axes
                min_score_val = min(df_res_viz['state_anomaly_score'].min(skipna=True), df_res_viz['duration_z_score'].abs().min(skipna=True)) if pd.Series([df_res_viz['state_anomaly_score'].min(skipna=True), df_res_viz['duration_z_score'].abs().min(skipna=True)]).notna().any() else 0
                max_score_val = max(df_res_viz['state_anomaly_score'].max(skipna=True), df_res_viz['duration_z_score'].abs().max(skipna=True)) if pd.Series([df_res_viz['state_anomaly_score'].max(skipna=True), df_res_viz['duration_z_score'].abs().max(skipna=True)]).notna().any() else 1
                
                # Create a second y-axis for state_anomaly_score
                ax2 = ax1.twinx()
                color_state_anomaly = 'tab:red'
                ax2.set_ylabel('State Anomaly Score (TFT MAE)', color=color_state_anomaly)
                ax2.plot(df_res_viz['timestamp_numeric'], df_res_viz['state_anomaly_score'].fillna(0), label='State Anomaly Score (TFT)', color=color_state_anomaly, linestyle='--', marker='x', markersize=3, alpha=0.6)
                ax2.axhline(y=state_anomaly_threshold, color=color_state_anomaly, linestyle=':', linewidth=1, label=f'State Thr={state_anomaly_threshold}')
                ax2.tick_params(axis='y', labelcolor=color_state_anomaly)
                ax2.legend(loc='upper right')
                if pd.Series([min_score_val, max_score_val]).notna().all() and max_score_val > min_score_val : ax2.set_ylim(min_score_val - 0.1 * (max_score_val-min_score_val), max_score_val + 0.1 * (max_score_val-min_score_val))


                # Create a third y-axis for duration_z_score
                ax3 = ax1.twinx()
                ax3.spines["right"].set_position(("outward", 60)) # Offset the third y-axis
                color_duration_anomaly = 'tab:green'
                ax3.set_ylabel('Abs Duration Z-Score', color=color_duration_anomaly)
                ax3.plot(df_res_viz['timestamp_numeric'], df_res_viz['duration_z_score'].abs().fillna(0), label='Abs Duration Z-Score', color=color_duration_anomaly, linestyle=':', marker='s', markersize=3, alpha=0.5)
                ax3.axhline(y=duration_z_threshold, color=color_duration_anomaly, linestyle='-.', linewidth=1, label=f'Dur. Thr={duration_z_threshold}')
                ax3.tick_params(axis='y', labelcolor=color_duration_anomaly)
                ax3.legend(loc='lower right')
                if pd.Series([min_score_val, max_score_val]).notna().all() and max_score_val > min_score_val : ax3.set_ylim(min_score_val - 0.1 * (max_score_val-min_score_val), max_score_val + 0.1 * (max_score_val-min_score_val))
                
                # Ensure y-axis limits for ax2 and ax3 are reasonable and potentially shared or scaled
                # This part can be tricky with three axes. For now, they scale independently based on their data.

                plt.title(f"Combined Anomaly Signals: {key.replace('_', ' ')}") # Cleaner title
                fig.tight_layout() 
                plt.grid(True, linestyle=':', alpha=0.3) # Apply grid to the primary axis (ax1)
                
                plot_filename = f"combined_timeseries_{key.replace(' ', '_').lower()}.eps"
                plot_path_ts = os.path.join(part3_plots_dir, plot_filename)
                plt.savefig(plot_path_ts, format='eps'); 
                print(f"Saved combined time series plot for {key} to {plot_path_ts}")
                figure_references_ts[key] = plot_filename # Store for later reference
                plt.show() 
            else:
                print(f"Skipping combined time series plot for {key} due to missing columns, empty data, or insufficient data points ({len(df_res_viz) if df_res_viz is not None else 0}).")
    else:
        print("No combined results data to plot for Part 3.")
        
    display(Markdown(f"--- Part 3: Duration Anomaly Detection Complete ---"))

# %% Cell to Execute Part 3
# This cell should be placed after the definitions above in your notebook.
# Make sure global variables like MAIN_OUTPUT_DIR, TFT_RESULTS_DIR, generated_tft_data_files 
# are set from Part 1 & 2 execution in the same notebook session.

if 'generated_tft_data_files' in locals() and \
   generated_tft_data_files and \
   generated_tft_data_files.get("normal") and \
   'TFT_RESULTS_DIR' in globals() and \
   os.path.exists(TFT_RESULTS_DIR): 

    # Define unique output directories for this specific Part 3 run
    # Changed directory name slightly to reflect new thresholds for clarity if re-running
    PART3_RESULTS_DIR_CURRENT_RUN = os.path.join(MAIN_OUTPUT_DIR, "part3_results_s05_z10") 
    PART3_PLOTS_DIR_CURRENT_RUN = os.path.join(MAIN_OUTPUT_DIR, "part3_plots_s05_z10")

    print(f"Part 3 results will be in: {PART3_RESULTS_DIR_CURRENT_RUN}")
    print(f"Part 3 plots will be in: {PART3_PLOTS_DIR_CURRENT_RUN}")

    # Make sure to pass the updated thresholds here
    run_part3_duration_anomaly_detection(
        processed_tft_files_dict=generated_tft_data_files, 
        part2_results_dir=TFT_RESULTS_DIR, 
        part3_results_dir=PART3_RESULTS_DIR_CURRENT_RUN,
        part3_plots_dir=PART3_PLOTS_DIR_CURRENT_RUN,
        state_anomaly_threshold=0.5,      # Updated state threshold
        duration_z_threshold=1.0          # Updated duration Z threshold
    )
else:
    display(Markdown("## Skipping Part 3: Duration Anomaly Detection"))
    reasons = []
    if not ('generated_tft_data_files' in locals() and generated_tft_data_files and generated_tft_data_files.get("normal")):
        reasons.append("Prerequisite data from Part 1 (`generated_tft_data_files`) is not available or normal data missing.")
    if not ('TFT_RESULTS_DIR' in globals() and os.path.exists(TFT_RESULTS_DIR)): 
        reasons.append(f"Part 2 results directory (TFT_RESULTS_DIR variable pointing to '{TFT_RESULTS_DIR if 'TFT_RESULTS_DIR' in globals() else 'Not Set'}') not found or not defined.")
    if reasons:
        print("Reason(s) for skipping Part 3:\n" + "\n".join(reasons))
    else: # Should not happen if the outer if condition failed
        print("An unknown issue prevented Part 3 from running.")